# Line Model — Baseline

In [2]:
%tb
import os, json, math, random, glob
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
from torch.optim import AdamW

from modules.Plotting import MetricLog, plot_metrics
from modules.HandTesting import hand_test_repl
from modules.BestModelSaver import BestModelSaver

import warnings
warnings.filterwarnings("ignore")

# WORKDIR = r'C:\Programing\code_autocomplete'
WORKDIR = r'C:\Users\Roman\Documents\Projects\code_autocomplete'
print(f"WORKDIR: {WORKDIR}")
LINE_MODEL_NAME = 'line_model_baseline'


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")
print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
print(f"Версия CUDA, под которую собран PyTorch: {torch.version.cuda}")
print(f"Количество GPU: {torch.cuda.device_count()}")

NameError: name 'WORKDIR' is not defined

WORKDIR: C:\Users\Roman\Documents\Projects\code_autocomplete
[Device] cuda
PyTorch версия: 2.6.0+cu124
CUDA доступна: True
Версия CUDA, под которую собран PyTorch: 12.4
Количество GPU: 1


## Tokenizer

In [3]:
# character-level BPE-lite

SPECIAL = {"<PAD>": 0, "<UNK>": 1, "<BOS>": 2, "<EOS>": 3}

class CodeTokenizer:
    """
    Simple sub-word tokenizer tailored for Python source code.
    Splits on whitespace/punctuation, keeps indentation tokens,
    and falls back to characters for unknowns.
    """
    PUNCT = set(",;&|~^@#")

    def __init__(self, vocab_size: int = 8000):
        self.vocab_size = vocab_size
        self.token2id: Dict[str, int] = dict(SPECIAL)
        self.id2token: Dict[int, str] = {v: k for k, v in SPECIAL.items()}
        self.built = False

    # ── build ──────────────────────────────────────────────
    def build(self, texts: List[str], min_freq: int = 3):
        freq: Dict[str, int] = defaultdict(int)
        for t in texts:
            for tok in self._raw_split(t):
                freq[tok] += 1
        sorted_tokens = sorted(freq.items(), key=lambda x: -x[1])
        for tok, cnt in sorted_tokens:
            if cnt < min_freq:
                break
            if tok not in self.token2id and len(self.token2id) < self.vocab_size:
                idx = len(self.token2id)
                self.token2id[tok] = idx
                self.id2token[idx] = tok
        # fill remaining slots with single chars
        for c in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789.,_ \t\n":  #for c in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_ \t\n":
            if c not in self.token2id and len(self.token2id) < self.vocab_size:
                idx = len(self.token2id)
                self.token2id[c] = idx
                self.id2token[idx] = c
        self.built = True
        print(f"[Tokenizer] vocab_size={len(self.token2id)}")

    def _raw_split(self, text: str) -> List[str]:
        tokens = []
        for line in text.splitlines(keepends=True):
            # capture leading whitespace as indent token
            stripped = line.lstrip(" \t")
            indent = line[: len(line) - len(stripped)]
            for ch in indent:
                tokens.append(ch)
            # split remainder on punctuation / spaces
            buf = ""
            for ch in stripped:
                if ch in self.PUNCT or ch in " \t\n\r":
                    if ch == "\n" or ch.strip():
                        if buf:
                            tokens.append(buf)
                        tokens.append(ch)
                        continue
                    if buf:
                        buf += ch
                        tokens.append(buf)
                        buf = ""
                        tokens.append("\n")
                else:
                    buf += ch
            if buf:
                tokens.append(buf)
        return tokens

    def encode(self, text: str) -> List[int]:
        ids = [SPECIAL["<BOS>"]]
        for tok in self._raw_split(text):
            if tok in self.token2id:
                ids.append(self.token2id[tok])
            else:
                # char fallback
                for ch in tok:
                    ids.append(self.token2id.get(ch, SPECIAL["<UNK>"]))
        ids.append(SPECIAL["<EOS>"])
        return ids

    def decode(self, ids: List[int]) -> str:
        parts = []
        for i in ids:
            tok = self.id2token.get(i, "")
            if tok in SPECIAL:
                continue
            parts.append(tok)
        return "".join(parts)

    def save(self, path: str):
        with open(path, "w") as f:
            json.dump({"token2id": self.token2id}, f)

    @classmethod
    def load(cls, path: str) -> "CodeTokenizer":
        with open(path) as f:
            d = json.load(f)
        obj = cls()
        obj.token2id = {k: int(v) for k, v in d["token2id"].items()}
        obj.id2token = {v: k for k, v in obj.token2id.items()}
        obj.built = True
        return obj

    @property
    def pad_id(self):  return SPECIAL["<PAD>"]
    @property
    def eos_id(self):  return SPECIAL["<EOS>"]
    @property
    def bos_id(self):  return SPECIAL["<BOS>"]
    @property
    def vocab(self):   return len(self.token2id)

## Datasets

In [4]:
def load_files(data_dir: str, max_files: int = 0) -> List[str]:
    """Load .py / .txt files from a directory tree."""
    patterns = ["**/*.py", "**/*.txt"]
    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(data_dir, pat), recursive=True))
    if max_files:
        files = files[:max_files]
    texts = []
    for fp in files:
        try:
            texts.append(Path(fp).read_text(errors="replace"))
        except Exception:
            pass
    print(f"[Data] loaded {len(texts)} files from {data_dir}")
    return texts



class LineDataset(Dataset):
    """
    One sample = (prefix_tokens, full_line_tokens).
    The model learns to predict the rest of the current line given a prefix.
    """
    def __init__(self, texts: List[str], tokenizer: CodeTokenizer,
                 max_prefix: int = 96, max_line: int = 64):
        self.samples: List[Tuple[List[int], List[int]]] = []
        print(len([line for text in texts for line in text.splitlines()]))
        for text in texts:
            for line in text.splitlines():
                commentary_pos = line.find('#') 
                if commentary_pos != -1 and not line[commentary_pos - 1] in ['\'', '"']:
                    # print(line)
                    line = line[:line.find('#')]
                line = line.rstrip()
                if len(line.strip()) < 10:
                    continue
                full = tokenizer.encode(line)
                if len(full) < 4:
                    continue
                split = random.randint(2, max(2, len(full) - 2))
                prefix = full[:split][-max_prefix:]
                target = full[split:][:max_line]
                target.append(tokenizer.eos_id)
                self.samples.append((prefix, target))
        print(f"[LineDataset] {len(self.samples)} samples")

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        return self.samples[i]


def collate_line(batch, pad_id: int):
    prefixes, targets = zip(*batch)
    max_p = max(len(p) for p in prefixes)
    max_t = max(len(t) for t in targets)
    P = torch.full((len(batch), max_p), pad_id, dtype=torch.long)
    T = torch.full((len(batch), max_t), pad_id, dtype=torch.long)
    for i, (p, t) in enumerate(zip(prefixes, targets)):
        P[i, :len(p)] = torch.tensor(p)
        T[i, :len(t)] = torch.tensor(t)
    return P, T

## Models

In [5]:
@dataclass
class ModelCfg:
    vocab: int = 8000
    d_model: int = 256
    n_heads: int = 8
    n_layers: int = 4
    d_ff: int = 1024
    max_len: int = 256
    dropout: float = 0.1


class PositionalEncoding(nn.Module):
    def __init__(self, d: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.drop(x + self.pe[:, :x.size(1)])



class LineModel(nn.Module):
    """
    Encoder-Decoder Transformer for seq2seq line completion.
    Encoder: reads prefix.  Decoder: generates rest of line.
    """
    def __init__(self, cfg: ModelCfg):
        super().__init__()
        self.cfg = cfg
        self.enc_emb  = nn.Embedding(cfg.vocab, cfg.d_model, padding_idx=0)
        self.dec_emb  = nn.Embedding(cfg.vocab, cfg.d_model, padding_idx=0)
        self.enc_pos  = PositionalEncoding(cfg.d_model, cfg.max_len, cfg.dropout)
        self.dec_pos  = PositionalEncoding(cfg.d_model, cfg.max_len, cfg.dropout)
        self.transformer = nn.Transformer(
            cfg.d_model, cfg.n_heads, cfg.n_layers, cfg.n_layers,
            cfg.d_ff, cfg.dropout, batch_first=True, norm_first=True
        )
        self.head = nn.Linear(cfg.d_model, cfg.vocab, bias=False)
        self.dec_emb.weight = self.head.weight

    def forward(self, src: torch.Tensor, tgt: torch.Tensor,
                src_key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        T = tgt.size(1)
        causal = nn.Transformer.generate_square_subsequent_mask(T, device=src.device)
        enc_out = self.transformer.encoder(
            self.enc_pos(self.enc_emb(src)),
            src_key_padding_mask=src_key_padding_mask
        )
        dec_out = self.transformer.decoder(
            self.dec_pos(self.dec_emb(tgt)),
            enc_out,
            tgt_mask=causal,
            tgt_is_causal=True,
            memory_key_padding_mask=src_key_padding_mask
        )
        return self.head(dec_out)

    @torch.no_grad()
    def generate(self, prefix_ids: List[int], max_new: int = 64,
                 temperature: float = 0.7, top_k: int = 40,
                 tokenizer: Optional[CodeTokenizer] = None) -> List[int]:
        self.eval()
        dev = next(self.parameters()).device
        src = torch.tensor([prefix_ids], dtype=torch.long, device=dev)
        dec_ids = [SPECIAL["<BOS>"]]
        out_ids = []
        for _ in range(max_new):
            tgt = torch.tensor([dec_ids], dtype=torch.long, device=dev)
            logits = self(src, tgt)[0, -1] / temperature
            if top_k:
                topk_v, _ = torch.topk(logits, top_k)
                logits[logits < topk_v[-1]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1).item()
            if nxt == SPECIAL["<EOS>"]:
                break
            dec_ids.append(nxt)
            out_ids.append(nxt)
        return out_ids

## Training loop

In [6]:
def _clip_norm(model: nn.Module, max_norm: float = 1.0) -> float:
    return nn.utils.clip_grad_norm_(model.parameters(), max_norm).item()


def train_line_model(
    model:      LineModel,
    train_dl:   DataLoader,
    val_dl:     DataLoader,
    epochs:     int,
    lr:         float,
    device:     torch.device,
    saver:      BestModelSaver,
    log:        MetricLog,
    plot_dir:   str,
):
    tqdm.write(f"[Line] DataLoader — {len(train_dl)} train batches, "
               f"{len(val_dl)} val batches")

    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 20)
    crit = nn.CrossEntropyLoss(ignore_index=SPECIAL["<PAD>"])

    for ep in range(1, epochs + 1):
        # ── train ────────────────────────────────────────────
        print(f"Epoch {ep}")
        model.train()
        t_loss = t_acc = t_steps = 0
        for src, tgt in tqdm(train_dl, desc=f"[Token] Epoch {ep}/{epochs} train",
                 leave=False, unit="batch"):
            src, tgt = src.to(device), tgt.to(device)
            pad_mask = (src == SPECIAL["<PAD>"])
            dec_in = tgt[:, :-1]
            dec_out = tgt[:, 1:]
            logits = model(src, dec_in, src_key_padding_mask=pad_mask)
            loss = crit(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1))
            opt.zero_grad()
            loss.backward()
            gn = _clip_norm(model)
            opt.step()
            t_loss += loss.item()
            preds = logits.argmax(-1)
            mask = (dec_out != SPECIAL["<PAD>"])
            t_acc += (preds[mask] == dec_out[mask]).float().mean().item()
            t_steps += 1

        tl = t_loss / t_steps
        ta = t_acc / t_steps

        # ── val ──────────────────────────────────────────────
        model.eval()
        v_loss = v_steps = 0
        with torch.no_grad():
            for src, tgt in tqdm(val_dl, desc=f"[Token] Epoch {ep}/{epochs} val  ",
                 leave=False, unit="batch"):
                src, tgt = src.to(device), tgt.to(device)
                pad_mask = (src == SPECIAL["<PAD>"])
                dec_in   = tgt[:, :-1]
                dec_out  = tgt[:, 1:]
                logits   = model(src, dec_in, src_key_padding_mask=pad_mask)
                loss     = crit(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1))
                v_loss  += loss.item()
                v_steps += 1
        vl = v_loss / v_steps if v_steps else tl
        sched.step()

        log.append(train_loss=tl, val_loss=vl,
                   train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
                   lr=opt.param_groups[0]["lr"],
                   token_acc=ta, grad_norm=gn)
        tqdm.write(
            f"[Line  ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
            f"  ppl={math.exp(min(vl,20)):.1f}  acc={ta:.3f}"
            f"  lr={opt.param_groups[0]['lr']:.2e}"
        )

        saver.save(model, vl, ep)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Epoch {ep}",
                         f"{plot_dir}/{LINE_MODEL_NAME}_ep{ep:02d}.png")

    plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Final", f"{plot_dir}/{LINE_MODEL_NAME}_final.png")

## Main

In [7]:
class Arguments():
    def __init__(self, data_dir: str = f"{WORKDIR}/Clean_Dataset", ckpt_dir: str = f"{WORKDIR}/checkpoints/{LINE_MODEL_NAME}",
                    plot_dir: str = F"{WORKDIR}/plots/{LINE_MODEL_NAME}", tokenizer: str = "tokenizer.json", 
                    epochs: int = 5, batch: int = 32, lr: float = 5e-4,
                    ctx: int = 128, d_model: int = 256, n_layers: int = 4,
                    n_heads: int = 8, vocab_size: int = 000, max_files: int = 0,
                    val_split: float = 0.1, seed: int = 42, for_usage: bool = False,
                    skip_token: bool = False, skip_line: bool = False, test: bool = False):
        self.data_dir = data_dir
        self.ckpt_dir = ckpt_dir
        self.plot_dir = plot_dir
        self.tokenizer = tokenizer
        self.epochs = epochs
        self.batch = batch
        self.lr = lr
        self.ctx = ctx
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.vocab_size = vocab_size
        self.max_files = max_files
        self.val_split = val_split
        self.seed = seed
        self.skip_token = skip_token
        self.skip_line = skip_line
        self.test = test
        self.for_usage = for_usage


def main():
    args = Arguments()
    args = Arguments(epochs=2, max_files=100)
    # args = Arguments(skip_line=True, max_files=100, epochs=1)
    # args = Arguments(max_files=100, epochs=2, skip_token=True, vocab_size=160000)
    # args = Arguments(skip_token=True)
    # args = Arguments(test=True)
    # args = Arguments(for_usage==True)
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(args.plot_dir,  exist_ok=True)

    # ── tokenizer ────────────────────────────────────────────
    if os.path.exists(args.tokenizer):
        print(f"[Tokenizer] loading {args.tokenizer}")
        tokenizer = CodeTokenizer.load(args.tokenizer)
    else:
        print("[Tokenizer] building from data …")
        texts = load_files(args.data_dir, args.max_files)
        tokenizer = CodeTokenizer(vocab_size=args.vocab_size)
        tokenizer.build(texts)
        tokenizer.save(args.tokenizer)

    cfg = ModelCfg(
        vocab=tokenizer.vocab, d_model=args.d_model,
        n_heads=args.n_heads,  n_layers=args.n_layers,
        d_ff=args.d_model * 4, max_len=args.ctx + 32,
    )

    # torch.serialization.add_safe_globals([ModelCfg])

    # ── test-only mode ───────────────────────────────────────
    if args.test:
        lm = LineModel(cfg).to(device)
        line_paths = sorted(glob.glob(str(Path(args.ckpt_dir) / LINE_MODEL_NAME + "_*.pt")))
        if line_paths:
            ck = torch.load(line_paths[0], map_location=device, weights_only=False)
            lm.load_state_dict(ck["model_state"])
            print(f"[Loaded] line model from {line_paths[0]}")
    
        hand_test_repl(None, lm, None, tokenizer, device)
        return
    

    # ── load data ────────────────────────────────────────────
    print("[Loading] Started loading")
    texts = load_files(args.data_dir, args.max_files)
    if not texts:
        print("[ERROR] no data files found. Please put .py files in --data_dir")
        return
    print("[Loading] Ended loading")

    random.shuffle(texts)
    split = max(1, int(len(texts) * (1 - args.val_split)))
    tr_txt = texts[:split]
    va_txt = texts[split:]
    
    # ── LINE MODEL ──────────────────────────────────────────
    if not args.skip_line:
        print("  Prepairing LINE model")

        tr_line_ds = LineDataset(tr_txt, tokenizer)
        va_line_ds = LineDataset(va_txt, tokenizer)
        collate = lambda b: collate_line(b, tokenizer.pad_id) 
        tr_line_dl = DataLoader(tr_line_ds, args.batch, shuffle=True,
                                # num_workers=0, pin_memory=True)
                                collate_fn=collate, num_workers=0, pin_memory=True)
        va_line_dl = DataLoader(va_line_ds, args.batch, shuffle=False,
                                # num_workers=0, pin_memory=True)
                                collate_fn=collate, num_workers=0, pin_memory=True)

        line_model = LineModel(cfg).to(device)
        n_params = sum(p.numel() for p in line_model.parameters() if p.requires_grad)
        print(f"[Line  Model] {n_params/1e6:.2f}M parameters")

        line_saver = BestModelSaver(args.ckpt_dir, LINE_MODEL_NAME)
        line_log = MetricLog()
        print("  Training LINE model")
        train_line_model(line_model, tr_line_dl, va_line_dl, args.epochs, args.lr,
                         device, line_saver, line_log, args.plot_dir)

    # ── interactive test ─────────────────────────────────────
    hand_test_repl(None, line_model, None, tokenizer, device)


main()

[Tokenizer] loading tokenizer.json
[Loading] Started loading
[Data] loaded 100 files from C:\Users\Roman\Documents\Projects\code_autocomplete/Clean_Dataset
[Loading] Ended loading
  Prepairing LINE model
13354
[LineDataset] 8787 samples
1549
[LineDataset] 994 samples
[Line  Model] 7.38M parameters
  Training LINE model
[Line] DataLoader — 275 train batches, 32 val batches
Epoch 1


[Line  ep   1] train_loss=0.2229  val_loss=0.1474  ppl=1.2  acc=0.939  lr=2.62e-04
[Saver] saved ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\line_model_baseline\line_model_baseline_ep001_loss0.1474.pt  (val_loss=0.1474)
[Plot] saved → C:\Users\Roman\Documents\Projects\code_autocomplete/plots/line_model_baseline/line_model_baseline_ep01.png
Epoch 2


[Line  ep   2] train_loss=0.1544  val_loss=0.1463  ppl=1.2  acc=0.963  lr=2.50e-05
[Saver] saved ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\line_model_baseline\line_model_baseline_ep002_loss0.1463.pt  (val_loss=0.1463)
[Plot] saved → C:\Users\Roman\Documents\Projects\code_autocomplete/plots/line_model_baseline/line_model_baseline_ep02.png
[Plot] saved → C:\Users\Roman\Documents\Projects\code_autocomplete/plots/line_model_baseline/line_model_baseline_final.png
  Python Autocomplete — Interactive Test
  Commands: :line <prefix>  | :temp <float>  | :k <int> 
            :token <prefix> | :quit
